# E-Commerce Customer Intelligence & Churn Prediction

## Repeat Purchase Prediction

This notebook develops and evaluates machine learning models for predicting whether a customer will make another purchase within the defined future observation window.

### Modeling Approach

The project uses:

- Time-based train/test split
- Historical customer behavior features
- StandardScaler where appropriate
- SMOTE applied only inside the training pipeline
- RandomizedSearchCV
- Logistic Regression
- Random Forest
- XGBoost
- Recall, Precision, F1, ROC-AUC and PR-AUC
- Confusion Matrix
- ROC Curve
- Precision-Recall Curve

### Important Data Science Decisions

The test dataset represents the latest historical snapshot and remains completely untouched during model training and hyperparameter tuning.

SMOTE is applied only inside the training cross-validation pipeline to prevent synthetic samples from influencing the test evaluation.

Customer identifiers, dates and future information are excluded from model features.

In [1]:
import json
import warnings
from pathlib import Path

import joblib
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from imblearn.over_sampling import SMOTE
from imblearn.pipeline import Pipeline

from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler

from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier

from sklearn.model_selection import (
    RandomizedSearchCV,
    StratifiedKFold
)

from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    roc_auc_score,
    average_precision_score,
    confusion_matrix,
    classification_report,
    ConfusionMatrixDisplay,
    RocCurveDisplay,
    PrecisionRecallDisplay
)

from xgboost import XGBClassifier

warnings.filterwarnings("ignore")

pd.set_option("display.max_columns", None)
pd.set_option("display.max_rows", 100)

In [2]:
PROJECT_ROOT = Path.cwd().parent

TRAIN_FILE = (
    PROJECT_ROOT
    / "data"
    / "processed"
    / "repeat_purchase_train.csv"
)

TEST_FILE = (
    PROJECT_ROOT
    / "data"
    / "processed"
    / "repeat_purchase_test.csv"
)

MODEL_DIR = PROJECT_ROOT / "models"
REPORT_DIR = PROJECT_ROOT / "reports"

MODEL_DIR.mkdir(exist_ok=True)
REPORT_DIR.mkdir(exist_ok=True)

print("Project Root:")
print(PROJECT_ROOT)

print("\nTrain file:")
print(TRAIN_FILE)

print("\nTest file:")
print(TEST_FILE)

Project Root:
d:\Data scientist\E-Commerce Customer Intelligence & Churn Prediction

Train file:
d:\Data scientist\E-Commerce Customer Intelligence & Churn Prediction\data\processed\repeat_purchase_train.csv

Test file:
d:\Data scientist\E-Commerce Customer Intelligence & Churn Prediction\data\processed\repeat_purchase_test.csv


In [3]:
train_df = pd.read_csv(TRAIN_FILE)
test_df = pd.read_csv(TEST_FILE)

print("TRAIN DATA")
print(train_df.shape)

print("\nTEST DATA")
print(test_df.shape)

TRAIN DATA
(115063, 18)

TEST DATA
(55907, 18)


In [5]:
display(train_df.head())
display(train_df.info())

,customer_unique_id,snapshot_date,future_end_date,first_purchase_date,last_purchase_date,total_orders,total_revenue,average_order_value,recency_days,customer_lifetime_days,repeat_customer,purchase_frequency,mean_purchase_gap_days,median_purchase_gap_days,max_purchase_gap_days,purchase_gap_count,observation_window_days,repeat_purchase
0,ffff371b4d645b6ecea244b27531430a,2017-05-01 15:00:37,2017-10-28 15:00:37,2017-02-07 15:49:16,2017-02-07 15:49:16,1,112.46,112.46,82.966215,1.0,0,30.0,NaN,NaN,NaN,NaN,180,0
1,5353fecd3b6270bcef1daae093bb6e33,2017-05-01 15:00:37,2017-10-28 15:00:37,2017-04-25 18:03:03,2017-04-25 18:03:03,1,47.90,47.90,5.873310,1.0,0,30.0,NaN,NaN,NaN,NaN,180,0
2,535d35b2288cbdeb6a7ba74ca2173213,2017-05-01 15:00:37,2017-10-28 15:00:37,2017-02-08 14:36:40,2017-02-08 14:36:40,1,194.01,194.01,82.016632,1.0,0,30.0,NaN,NaN,NaN,NaN,180,0
3,536a45120c7e443ad9d244872e8e06a4,2017-05-01 15:00:37,2017-10-28 15:00:37,2017-04-03 12:13:04,2017-04-03 12:13:04,1,21.87,21.87,28.116354,1.0,0,30.0,NaN,NaN,NaN,NaN,180,0
4,53730ba9200fe3371560f32ee5b17424,2017-05-01 15:00:37,2017-10-28 15:00:37,2017-02-14 22:09:54,2017-02-14 22:09:54,1,41.70,41.70,75.701887,1.0,0,30.0,NaN,NaN,NaN,NaN,180,0


<class 'pandas.DataFrame'>
RangeIndex: 115063 entries, 0 to 115062
Data columns (total 18 columns):
 #   Column                    Non-Null Count   Dtype  
---  ------                    --------------   -----  
 0   customer_unique_id        115063 non-null  str    
 1   snapshot_date             115063 non-null  str    
 2   future_end_date           115063 non-null  str    
 3   first_purchase_date       115063 non-null  str    
 4   last_purchase_date        115063 non-null  str    
 5   total_orders              115063 non-null  int64  
 6   total_revenue             115063 non-null  float64
 7   average_order_value       115063 non-null  float64
 8   recency_days              115063 non-null  float64
 9   customer_lifetime_days    115063 non-null  float64
 10  repeat_customer           115063 non-null  int64  
 11  purchase_frequency        115063 non-null  float64
 12  mean_purchase_gap_days    2983 non-null    float64
 13  median_purchase_gap_days  2983 non-null    float64
 14 

None

In [6]:
TARGET = "repeat_purchase"

print("Training Target Distribution")
print(train_df[TARGET].value_counts())

print("\nTraining Target Percentage")
print(
    train_df[TARGET]
    .value_counts(normalize=True)
    .mul(100)
    .round(2)
)

Training Target Distribution
repeat_purchase
0    113393
1      1670
Name: count, dtype: int64

Training Target Percentage
repeat_purchase
0    98.55
1     1.45
Name: proportion, dtype: float64


In [7]:
print("Test Target Distribution")
print(test_df[TARGET].value_counts())

print("\nTest Target Percentage")
print(
    test_df[TARGET]
    .value_counts(normalize=True)
    .mul(100)
    .round(2)
)

Test Target Distribution
repeat_purchase
0    55252
1      655
Name: count, dtype: int64

Test Target Percentage
repeat_purchase
0    98.83
1     1.17
Name: proportion, dtype: float64


In [8]:
FORBIDDEN_COLUMNS = [
    "customer_unique_id",
    "snapshot_date",
    "future_end_date",
    "first_purchase_date",
    "last_purchase_date",
    "future_purchase_count",
    "churn_label",
    "repeat_purchase"
]

available_columns = [
    col
    for col in train_df.columns
    if col not in FORBIDDEN_COLUMNS
]

print("Candidate Model Features:")
for col in available_columns:
    print("-", col)

Candidate Model Features:
- total_orders
- total_revenue
- average_order_value
- recency_days
- customer_lifetime_days
- repeat_customer
- purchase_frequency
- mean_purchase_gap_days
- median_purchase_gap_days
- max_purchase_gap_days
- purchase_gap_count
- observation_window_days


In [9]:
FEATURE_COLUMNS = [
    "total_orders",
    "total_revenue",
    "average_order_value",
    "recency_days",
    "customer_lifetime_days",
    "repeat_customer",
    "purchase_frequency",
    "mean_purchase_gap_days",
    "median_purchase_gap_days",
    "max_purchase_gap_days",
    "purchase_gap_count",
    "observation_window_days"
]

print("FINAL MODEL FEATURES")
print("=" * 50)

for feature in FEATURE_COLUMNS:
    print(feature)

FINAL MODEL FEATURES
total_orders
total_revenue
average_order_value
recency_days
customer_lifetime_days
repeat_customer
purchase_frequency
mean_purchase_gap_days
median_purchase_gap_days
max_purchase_gap_days
purchase_gap_count
observation_window_days


In [10]:
missing_train_features = [
    col
    for col in FEATURE_COLUMNS
    if col not in train_df.columns
]

missing_test_features = [
    col
    for col in FEATURE_COLUMNS
    if col not in test_df.columns
]

print("Missing train features:", missing_train_features)
print("Missing test features:", missing_test_features)

Missing train features: []
Missing test features: []


In [11]:
X_train = train_df[FEATURE_COLUMNS].copy()
y_train = train_df[TARGET].astype(int)

X_test = test_df[FEATURE_COLUMNS].copy()
y_test = test_df[TARGET].astype(int)

X_train = X_train.apply(
    pd.to_numeric,
    errors="coerce"
)

X_test = X_test.apply(
    pd.to_numeric,
    errors="coerce"
)

print("X_train:", X_train.shape)
print("y_train:", y_train.shape)

print("X_test:", X_test.shape)
print("y_test:", y_test.shape)

X_train: (115063, 12)
y_train: (115063,)
X_test: (55907, 12)
y_test: (55907,)


In [12]:
print("Training missing values:")
display(X_train.isna().sum())

print("\nTesting missing values:")
display(X_test.isna().sum())

Training missing values:


total_orders                     0
total_revenue                    0
average_order_value              0
recency_days                     0
customer_lifetime_days           0
repeat_customer                  0
purchase_frequency               0
mean_purchase_gap_days      112080
median_purchase_gap_days    112080
max_purchase_gap_days       112080
purchase_gap_count          112080
observation_window_days          0
dtype: int64


Testing missing values:


total_orders                    0
total_revenue                   0
average_order_value             0
recency_days                    0
customer_lifetime_days          0
repeat_customer                 0
purchase_frequency              0
mean_purchase_gap_days      54271
median_purchase_gap_days    54271
max_purchase_gap_days       54271
purchase_gap_count          54271
observation_window_days         0
dtype: int64

In [13]:
cv = StratifiedKFold(
    n_splits=5,
    shuffle=True,
    random_state=42
)

print("5-Fold Stratified Cross Validation configured.")

5-Fold Stratified Cross Validation configured.


In [14]:
logistic_pipeline = Pipeline(
    steps=[
        (
            "imputer",
            SimpleImputer(strategy="median")
        ),
        (
            "scaler",
            StandardScaler()
        ),
        (
            "smote",
            SMOTE(random_state=42)
        ),
        (
            "model",
            LogisticRegression(
                max_iter=3000,
                random_state=42
            )
        )
    ]
)

In [15]:
logistic_params = {
    "smote__sampling_strategy": [
        0.02,
        0.05,
        0.10,
        0.20,
        0.30,
        0.50,
        0.75,
        1.0
    ],

    "smote__k_neighbors": [
        3,
        5,
        7
    ],

    "model__C": [
        0.001,
        0.01,
        0.1,
        1,
        10,
        100
    ],

    "model__solver": [
        "liblinear",
        "lbfgs"
    ],

    "model__class_weight": [
        None,
        "balanced"
    ]
}

In [17]:
logistic_search = RandomizedSearchCV(
    estimator=logistic_pipeline,
    param_distributions=logistic_params,
    n_iter=30,
    scoring="recall",
    cv=cv,
    random_state=42,
    n_jobs=-1,
    verbose=1,
    return_train_score=True
)

logistic_search.fit(
    X_train,
    y_train
)

print("Best CV Recall:")
print(logistic_search.best_score_)

print("\nBest Parameters:")
print(logistic_search.best_params_)

Fitting 5 folds for each of 30 candidates, totalling 150 fits
Best CV Recall:
0.5221556886227544

Best Parameters:
{'smote__sampling_strategy': 0.3, 'smote__k_neighbors': 7, 'model__solver': 'lbfgs', 'model__class_weight': 'balanced', 'model__C': 1}


In [18]:
logistic_model = logistic_search.best_estimator_

logistic_pred = logistic_model.predict(X_test)

logistic_prob = (
    logistic_model
    .predict_proba(X_test)[:, 1]
)

In [19]:
def evaluate_model(
    model_name,
    y_true,
    y_pred,
    y_prob
):
    
    metrics = {
        "Model": model_name,
        "Accuracy": accuracy_score(y_true, y_pred),
        "Precision": precision_score(
            y_true,
            y_pred,
            zero_division=0
        ),
        "Recall": recall_score(
            y_true,
            y_pred,
            zero_division=0
        ),
        "F1": f1_score(
            y_true,
            y_pred,
            zero_division=0
        ),
        "ROC-AUC": roc_auc_score(
            y_true,
            y_prob
        ),
        "PR-AUC": average_precision_score(
            y_true,
            y_prob
        )
    }
    
    print("=" * 70)
    print(model_name)
    print("=" * 70)
    
    for key, value in metrics.items():
        if key != "Model":
            print(f"{key:<12}: {value:.4f}")
    
    print("\nConfusion Matrix:")
    print(confusion_matrix(y_true, y_pred))
    
    print("\nClassification Report:")
    print(
        classification_report(
            y_true,
            y_pred,
            digits=4,
            zero_division=0
        )
    )
    
    return metrics

In [21]:
logistic_metrics = evaluate_model(
    "SMOTE Logistic Regression",
    y_test,
    logistic_pred,
    logistic_prob
)

SMOTE Logistic Regression
Accuracy    : 0.6755
Precision   : 0.0166
Recall      : 0.4595
F1          : 0.0321
ROC-AUC     : 0.5954
PR-AUC      : 0.0292

Confusion Matrix:
[[37462 17790]
 [  354   301]]

Classification Report:
              precision    recall  f1-score   support

           0     0.9906    0.6780    0.8050     55252
           1     0.0166    0.4595    0.0321       655

    accuracy                         0.6755     55907
   macro avg     0.5036    0.5688    0.4186     55907
weighted avg     0.9792    0.6755    0.7960     55907

